In [1]:
# conda activate genomic_tools

import os
import glob
import json
import pickle
import pandas as pd
import bioframe as bf
from collections import defaultdict

pd.set_option('display.max_columns', None)

In [2]:
with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

## Prep data from UCSC genome browser tracks

In [3]:
track_df_list = []
keep = ["unipLocTransMemb", "unipOther", "unipLocSignal", "unipModif", "unipRepeat", 
        "unipLocCytopl", "unipChain", "unipLocExtra", "unipDomain", "unipStruct", 
        "unipDisulfBond", "unipInterest"]
track_dir = "/mnt/lareaulab/reliscu/data/UCSC/hg38/genome_tracks/uniprot"

for file in glob.glob(f"{track_dir}/*.bed"):
    file_name = file.split("/")[-1].split(".bb.bed")[0]
    if file_name in keep:
        track_df = pd.read_csv(file, sep="\t")
        track_df.insert(0, "file_name", file_name)
        track_df_list.append(track_df)

/tmp/ipykernel_2339317/3072133374.py:10: DtypeWarning: Columns (0: pmids) have mixed types. Specify dtype option on import or set low_memory=False.
  track_df = pd.read_csv(file, sep="\t")
/tmp/ipykernel_2339317/3072133374.py:10: DtypeWarning: Columns (0: longName, 1: syns) have mixed types. Specify dtype option on import or set low_memory=False.
  track_df = pd.read_csv(file, sep="\t")


## Prep InterProScan results

In [ ]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins_v2.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [6]:
interproscan_results.shape

(444449, 15)

In [7]:
interproscan_results.value_counts("signature_description")

signature_description
-                                              108487
Consensus disorder prediction                   54951
Transmembrane region                            13408
Coil                                            10280
Non cytoplasmic domain                           9707
                                                ...  
TARBP1 domain                                       1
SpoU rRNA Methylase family                          1
RNA methyltransferase TRMH family profile           1
N-terminal domain of synaptotagmin-1 and -2         1
Signal recognition particle 19 kDa protein          1
Name: count, Length: 24477, dtype: int64

In [8]:
# Parse the PIRSR data

with open("data/interproscan/interpro_data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [9]:
interproscan_results = interproscan_results.merge(
    pirsr_df, left_on="signature_accession", right_on="accession", how="left"
)

In [10]:
interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'signature_description'] = \
    interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'label']
interproscan_results.loc[interproscan_results['analysis'] == "DeepTMHMM", 'signature_description'] = \
    interproscan_results.loc[interproscan_results['analysis'] == "DeepTMHMM", 'signature_accession']
interproscan_results.loc[interproscan_results['analysis'] == "TMbed", 'signature_description'] = \
    interproscan_results.loc[interproscan_results['analysis'] == "TMbed", 'signature_accession']  

In [11]:
interproscan_results.head()

,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,stop,score,status,date,interpro_accession,interpro_description,go_annotations,pathways,accession,scope,TR,label,condition,desc,group
0,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:6.10.140.620,-,283,316,1.1E-25,CATH-Gene3D,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:3.30.200.20,Phosphorylase Kinase; domain 1,1,93,1.7E-39,CATH-Gene3D,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:1.10.510.10,Transferase(Phosphotransferase) domain 1,94,282,3.5E-66,CATH-Gene3D,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-Gene3D,G3DSA:3.10.450.50,-,334,486,6.9E-75,CATH-Gene3D,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENST00000508738,AE7E2D8B3DA6C8CA5D95C12174AD0E55,489,CATH-FunFam,G3DSA:3.10.450.50:FF:000001,calcium/calmodulin-dependent protein kinase ty...,340,486,1.2E-92,CATH-FunFam,19-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Map events to domains

#### Uniprot tracks overlapping splicing events

In [ ]:
FLANK = 600

def flatten_exon_info(data: dict, flank=600):
    rows = []
    for ev, rec in data.items():
        meta = rec["meta"]

        def emit(type_name, d, start=None, end=None, flanked_start=None, flanked_end=None):
            row = {
                "event_id": ev,
                "chrom": meta["chrom"],
                "strand": meta["strand"],
                "gene": meta["gene"],
                "meta_es": meta["es"],
                "meta_ee": meta["ee"],
                "type": type_name,
            }
            row.update(d)  # transcript_id, exon_cds_start/end, frame_preserving, etc.
            row['start'] = start
            row['end'] = end
            row['flanked_start'] = flanked_start
            row['flanked_end'] = flanked_end 
            rows.append(row)
        
        d = rec.get("inclusion")
        flanked_start = d['exon_cds_start'] - flank
        flanked_end = d['exon_cds_end'] + flank 
        emit("inclusion", d, 
            flanked_start=flanked_start,
            flanked_end=flanked_end
        )
                
        for type_name in ("exon_diff_boundary_siblings", "exon_diff_junction_siblings"):
            d = rec.get(type_name)
            if d and d.get("transcript_id") is not None:
                flanked_start = d['exon_cds_start'] - flank
                flanked_end = d['exon_cds_end'] + flank 
                emit(type_name, d, 
                    start=d['exon_cds_start'], 
                    end=d['exon_cds_end'],
                    flanked_start=flanked_start,
                    flanked_end=flanked_end
                )

        # real_skip: use flanked window around the parent exon
        d = rec.get("real_skip")
        if d and d.get("transcript_id") is not None:
            es = rec['inclusion']['exon_cds_start']
            ee = rec['inclusion']['exon_cds_end']
            flanked_start = es - flank
            flanked_end = ee + flank 
            emit("real_skip", d,
                start=es, end=ee,
                flanked_start=flanked_start,
                flanked_end=flanked_end
            )
        
    df = pd.DataFrame(rows)
    df["flanked_start"] = df["flanked_start"].astype(int)
    df["flanked_end"] = df["flanked_end"].astype(int)

    return df

In [ ]:
long_df = flatten_exon_info(event_protein_map)

cols = [
    "file_name", "chrom", "chromStart", "chromEnd", "name", "status", "annotationType", "position"
]

results = []
for track_df in track_df_list:
    # t = track_df.rename(columns={"chromStart": "start", "chromEnd": "end"})
    t = track_df.loc[:, cols]
    overlapped = bf.overlap(
        long_df, t,
        cols1=("chrom", "flanked_start", "flanked_end"),
        cols2=("chrom", "chromStart", "chromEnd"),
        suffixes=("", "_"),
        how="inner"
    )
    results.append(overlapped)

uniprot_merged = pd.concat(results, ignore_index=True)

In [14]:
uniprot_merged[(uniprot_merged['file_name_'] == "unipLocSignal") & (uniprot_merged['status_'].str.contains("Manually"))].head()

,event_id,chrom,strand,gene,meta_es,meta_ee,type,transcript_id,aa_start,aa_end,exon_cds_start,exon_cds_end,frame_preserving,clean_start,clean_end,start,end,flanked_start,flanked_end,excluded_transcript_types,excluded_transcript_tags,file_name_,chrom_,chromStart_,chromEnd_,name_,status_,annotationType_,position_
30301,ENSG00000134470_ProteinCoding_4,chr10,-,ENSG00000134470,5966145,5966339,inclusion,ENST00000528354,29.0,94.0,5966145.0,5966339.0,True,False,False,NaN,NaN,5965545,5966939,NaN,NaN,unipLocSignal,chr10,5966337,5977492,Signal peptide,Manually reviewed (Swiss-Prot),signal peptide,amino acids 1-30 on protein Q13261
30302,ENSG00000134470_ProteinCoding_4,chr10,-,ENSG00000134470,5966145,5966339,real_skip,ENST00000397250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5966145.0,5966339.0,5965545,5966939,{protein_coding_CDS_not_defined},{},unipLocSignal,chr10,5966337,5977492,Signal peptide,Manually reviewed (Swiss-Prot),signal peptide,amino acids 1-30 on protein Q13261
30305,ENSG00000150201_ProteinCoding_1,chr10,+,ENSG00000150201,43373515,43373783,inclusion,ENST00000476166,0.0,12.0,43373747.0,43373783.0,False,True,False,NaN,NaN,43373147,43374383,NaN,NaN,unipLocSignal,chr10,43373746,43374492,Signal peptide,Manually reviewed (Swiss-Prot),signal peptide,amino acids 1-20 on protein P59646
30307,ENSG00000180644_ProteinCoding_1,chr10,-,ENSG00000180644,70600364,70600906,inclusion,ENST00000638674,0.0,179.0,70600364.0,70600902.0,False,True,False,NaN,NaN,70599764,70601502,NaN,NaN,unipLocSignal,chr10,70600839,70600902,Signal peptide,Manually reviewed (Swiss-Prot),signal peptide,amino acids 1-21 on protein P14222
30308,ENSG00000122861_ProteinCoding_1,chr10,+,ENSG00000122861,73912041,73912068,inclusion,ENST00000372764,19.0,28.0,73912041.0,73912068.0,False,True,False,NaN,NaN,73911441,73912668,NaN,NaN,unipLocSignal,chr10,73911555,73912043,Signal peptide,Manually reviewed (Swiss-Prot),signal peptide,amino acids 1-20 on protein P00749


In [15]:
uniprot_by_event = {
    event_id: grp
    for event_id, grp in uniprot_merged.groupby("event_id")  # adjust column name — see note below
}

In [16]:
len(uniprot_by_event)

12498

In [17]:
pickle.dump(uniprot_by_event, open("data/uniprot_by_event.pkl", "wb"))

#### IntroProScan results overlapping splicing events

In [18]:
with open("data/gencode.v46.annotation_cds_by_transcript.pkl", "rb") as f:
    cds_by_transcript = pickle.load(f)

In [70]:
def near_exon(exon_start, exon_end, scan_start, scan_end, window=600):
    """Returns boolean mask for features within window of the exon."""
    return (
        (exon_end  >= scan_start - window) &
        (exon_start <= scan_end + window)
    )
    
def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]

def rel_cds_to_genome(cds_obj, strand):
    """
    Build a list of exon segments mapping between genomic coordinates
    and CDS-relative coordinates, in transcript (5'->3') order.

    Each segment is a dict:
        genome_start, genome_end : genomic coordinates (always genome_start <= genome_end)
        rel_start, rel_end       : CDS-relative coordinates (always rel_start <= rel_end)
    """
    cds = _cds_rows(cds_obj)

    # order exons in transcript (5'->3') order
    if strand == '-':
        cds = sorted(cds, key=lambda c: c['start'], reverse=True)
    else:
        cds = sorted(cds, key=lambda c: c['start'])

    segments = []
    cds_start = 0
    for c in cds:
        length = c['end'] - c['start'] + 1
        segments.append({
            'genome_start': c['start'],
            'genome_end': c['end'],
            'rel_start': cds_start,
            'rel_end': cds_start + length - 1,
        })
        cds_start += length

    return segments

def map_aa_to_genome(rel_to_genome, aa_start, aa_end, strand):
    result = []
    # convert AA position to relative CDS position
    # note: subtract 1 to convert from 1-indexed AA to 0-indexed
    cds_start = (aa_start - 1) * 3
    cds_end = (aa_end - 1) * 3 + 2
    for seg in rel_to_genome:
        # seg contains mapping from relative CDS position to genome position
        overlap_start = max(cds_start, seg['rel_start'])
        overlap_end = min(cds_end, seg['rel_end'])
        if overlap_start <= overlap_end:
            if strand == '-':
                # rel increases as genome decreases
                g_start = seg['genome_end'] - (overlap_end - seg['rel_start'])
                g_end = seg['genome_end'] - (overlap_start - seg['rel_start'])
            else:
                # rel increases as genome increases
                g_start = seg['genome_start'] + (overlap_start - seg['rel_start'])
                g_end = seg['genome_start'] + (overlap_end - seg['rel_start'])
            result.append((g_start, g_end))
    return result

def process_sibling(rec, key, ipr_grouped, cds_by_transcript, strand):
    entry = rec.get(key)
    if not entry or entry.get('transcript_id') is None:
        return None
    t = entry['transcript_id']
    sib_df = ipr_grouped.get(t)
    if sib_df is None:
        return None

    cds_obj = cds_by_transcript[t]
    cds_to_genome = rel_cds_to_genome(cds_obj, strand)

    genome_coords = []
    for _, row in sib_df.iterrows():
        aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
        scan_start = min(min(x) for x in aa_to_genome)
        scan_end = max(max(x) for x in aa_to_genome)
        if near_exon(entry['exon_cds_start'], entry['exon_cds_end'], scan_start, scan_end):
            genome_coords.append(aa_to_genome)
        else:
            genome_coords.append(None)

    sib_df = sib_df.assign(
        aa_start=entry['aa_start'],
        aa_end=entry['aa_end'],
        exon_cds_start=entry['exon_cds_start'],
        exon_cds_end=entry['exon_cds_end'],
        excluded_transcript_types=[entry['excluded_transcript_types']] * len(sib_df),
        excluded_transcript_tags=[entry['excluded_transcript_tags']] * len(sib_df),
        genome_coords=genome_coords
    )
    sib_df = sib_df[sib_df['genome_coords'].notna()]
    
    return sib_df.reset_index(drop=True) if not sib_df.empty else None

In [87]:
# map interproscan results to their corresponding transcript (representing the splicing event of interest)
cols = ['protein_accession', 'sequence_length', 'analysis', 
        'signature_description', 'start', 'stop', 'interpro_description']
ipr = interproscan_results.loc[:, cols]
ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

interproscan_by_event = defaultdict(dict)

for ev, rec in event_protein_map.items():
    
    rec_incl = rec['inclusion']
    incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
    if incl_df is None:
        continue
    
    strand = rec['meta']['strand']
    cds_obj = cds_by_transcript[rec_incl['transcript_id']]
    cds_to_genome = rel_cds_to_genome(cds_obj, strand)
    
    # convert AA pos to genome position
    genome_coords = []
    for _, row in incl_df.iterrows():
        aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
        scan_start = min(min(x) for x in aa_to_genome)
        scan_end = max(max(x) for x in aa_to_genome)
        if near_exon(rec_incl['exon_cds_start'], rec_incl['exon_cds_end'], scan_start, scan_end): 
            genome_coords.append(aa_to_genome)
        else:
            genome_coords.append(None)
    
    incl_cols = ['aa_start', 'aa_end', 
                'exon_cds_start', 'exon_cds_end',
                'frame_preserving', 'clean_start', 'clean_end']
        
    incl_df = incl_df.assign(
        **{col: rec_incl[col] for col in incl_cols},
        genome_coords=genome_coords
    ).reset_index(drop=True)
         
    incl_df = incl_df[incl_df['genome_coords'].notna()]
        
    if not incl_df.empty:
        interproscan_by_event[ev] = {
            'meta': rec['meta'],
            'inclusion': 
               incl_df.assign(
                    excluded_transcript_types=None,
                    excluded_transcript_tags=None
                ), 
            'real_skip': None,
            'exon_diff_junction_siblings': None,
            'exon_diff_boundary_siblings': None
        }
   
        ############################################################################################  
        # real skip
        if rec.get('real_skip'):
            t = rec['real_skip']['transcript_id'] 
            if t is not None:
                skip_df = ipr_grouped.get(t)
                if skip_df is not None:
                    cds_obj = cds_by_transcript[t]
                    cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                    
                    # convert AA pos to genome position
                    genome_coords = []
                    for _, row in skip_df.iterrows():
                        aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                        scan_start = min(min(x) for x in aa_to_genome)
                        scan_end = max(max(x) for x in aa_to_genome)
                        # real_skip: no exon coords (it's skipped) — use window around the parent exon
                        if near_exon(rec_incl['exon_cds_start'], rec_incl['exon_cds_end'], scan_start, scan_end): 
                            genome_coords.append(aa_to_genome)
                        else:
                            genome_coords.append(None)
                            
                    skip_df = skip_df.assign(
                        exon_cds_start=rec['meta']['es'],
                        exon_cds_end=rec['meta']['ee'],
                        genome_coords=genome_coords,
                        excluded_transcript_types=[rec['real_skip']['excluded_transcript_types']] * len(skip_df),
                        excluded_transcript_tags=[rec['real_skip']['excluded_transcript_tags']] * len(skip_df),
                    )
                    skip_df = skip_df[skip_df['genome_coords'].notna()]
                        
                    if not skip_df.empty:
                        interproscan_by_event[ev]['real_skip'] = skip_df.reset_index(drop=True)
        
        ############################################################################################      
        # junction sibling & boundary siblingss
        
        interproscan_by_event[ev]['exon_diff_junction_siblings'] = \
            process_sibling(rec, "exon_diff_junction_siblings", ipr_grouped, cds_by_transcript, strand)
        interproscan_by_event[ev]['exon_diff_boundary_siblings'] = \
            process_sibling(rec, "exon_diff_boundary_siblings", ipr_grouped, cds_by_transcript, strand)        

In [74]:
len(interproscan_by_event)

12237

In [75]:
pickle.dump(interproscan_by_event, open("data/interproscan_by_event.pkl", "wb"))

### Debug

In [ ]:
# ev = "ENSG00000285043_ProteinCoding_1"
# rec = event_protein_map[ev]

In [ ]:
# # map interproscan results to their corresponding transcript (representing the splicing event of interest)
# analyses_to_exclude = ['NCBIFAM', 'SFLD']
# columns = ['protein_accession', 'sequence_length', 'analysis', 
#            'signature_description', 'start', 'stop', 'interpro_description']
# ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

# ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

# event_interproscan_map = defaultdict(dict)

# rec_incl = rec['inclusion']
# incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
# if incl_df is None:
#     continue
# overlap_df = incl_df[
#     (incl_df['start'] <= rec_incl['aa_end']) &
#     (incl_df['stop']  >= rec_incl['aa_start'])
# ]



# strand = rec['meta']['strand']

In [ ]:
# # junction siblings

# for sib in rec['exon_diff_junction_siblings']:
#     t = sib['transcript_id']
#     sib_df = ipr_grouped.get(t) 
#     if sib_df is None:
#         continue
#     sib_overlap = sib_df[
#         (sib_df['start'] <= sib['aa_end']) &
#         (sib_df['stop']  >= sib['aa_start'])
#     ]
#     break


In [ ]:
# # save AA position of protein domains in terms of genomic coordinates
# cds_obj = cds_by_transcript[t]
# cds_to_genome = rel_cds_to_genome(cds_obj, strand)
# genome_coords = []
# for idx, row in sib_overlap.iterrows():
#     aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
#     genome_coords.append(aa_to_genome)

### End debug

## Merge results with cell type-specific splicing events

In [91]:
def combine_excluded_sets(x):
    combined = set()
    for s in x:
        if isinstance(s, set):
            combined |= s
    combined.discard('')
    return ' | '.join(sorted(str(v) for v in combined))

In [95]:
uniprot_cols = [
    "event_id", "is_specific", "r", "fdr", "type", "Gene", "transcript_id", "chrom", "strand",
    "exon_len", "frame_preserving", "clean_start", "clean_end", "aa_start", "aa_end", 
    "exon_cds_start", "exon_cds_end", "flanked_start", "flanked_end", 
    "excluded_transcript_types", "excluded_transcript_tags", 
    "file_name_", "chromStart_", "chromEnd_", "name_", "annotationType_", "status_"
]

interproscan_by_ct = dict() 
interproscan_by_ct_summary = dict()

uniprot_by_ct = dict()
uniprot_by_ct_summary = dict()

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ct = file.split("_exons.csv")[0]
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)

        print(ct)
        ######################## UNIPROT ######################### 
        uniprot_df = uniprot_merged.merge(
            signif_events_df.drop(columns=["strand"]), left_on="event_id", right_index=True
        )
        uniprot_by_ct[ct] = uniprot_df[uniprot_cols]

        uniprot_by_ct_summary[ct] = \
            uniprot_by_ct[ct].groupby(["event_id", "type", "transcript_id"]).agg(
                r=('r', lambda x: ' | '.join(map(str, x.unique()))),
                is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
                Gene=('Gene', lambda x: ' | '.join(x.unique())),
                frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
                n_analyses=('file_name_', lambda x: len(x.unique())),
                analyses=('file_name_', lambda x: ' | '.join(x.unique())),
                annotation=('annotationType_', lambda x: ' | '.join(str(v) for v in x.unique() if pd.notna(v))),
                status=('status_', lambda x: ' | '.join(x.unique())),
                excluded_transcript_types=('excluded_transcript_types', combine_excluded_sets),
                excluded_transcript_tags=('excluded_transcript_tags', combine_excluded_sets),
            ).reset_index()
        
        ######################### INTERPROSCAN #########################
        event_interproscan_dict = {
            ev: interproscan_by_event[ev] for ev in signif_events_df.index 
            if ev in interproscan_by_event
        }

        interpro_result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
             for ev, buckets in event_interproscan_dict.items()
             for bucket, df in buckets.items()
             if bucket != 'meta' and df is not None],
            ignore_index=True
        )
    
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in interpro_result.columns if c not in ('event_id', 'bucket')]
        interpro_result = interpro_result[cols]
        interproscan_by_ct[ct] = interpro_result.merge(signif_events_df, left_on="event_id", right_index=True)
        
        # summarize interpro results per splicing event
        interproscan_by_ct_summary[ct] = \
            interproscan_by_ct[ct].groupby(["event_id", "bucket", "protein_accession"]).agg(
                Gene=('Gene', lambda x: ' | '.join(x.unique())),
                r=('r', lambda x: ' | '.join(map(str, x.unique()))),
                is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
                frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
                n_analyses=('analysis', lambda x: len(x.unique())),
                analyses=('analysis', lambda x: ' | '.join(x.unique())),
                signature_descriptions=('signature_description', lambda x: ' | '.join(str(v) for v in x.unique() if pd.notna(v))),
                interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique())),
                excluded_transcript_types=('excluded_transcript_types', combine_excluded_sets),
                excluded_transcript_tags=('excluded_transcript_tags', combine_excluded_sets),
            ).reset_index()

Oligo
VLMC
Endo
Deep_layer_glutamatergic
Astro
OPC
Micro_PVM
All_Neuronal
All_GABAergic
Peri
CGE_Class
Upper_layer_glutamatergic


In [ ]:
with open("data/interproscan_by_ct.pkl", "wb") as file:
    pickle.dump(interproscan_by_ct, file)
    
with open("data/interproscan_by_ct_summary.pkl", "wb") as file:
    pickle.dump(interproscan_by_ct_summary, file)

In [ ]:
with open("data/uniprot_by_ct.pkl", "wb") as file:
    pickle.dump(uniprot_by_ct, file)
    
with open("data/uniprot_by_ct_summary.pkl", "wb") as file:
    pickle.dump(uniprot_by_ct_summary, file)

In [97]:
rec = interproscan_by_ct_summary['Deep_layer_glutamatergic']

rec[(rec['is_specific'] == "True") & (rec['bucket'] == "inclusion")].sort_values('n_analyses', ascending=False).head(10)

,event_id,bucket,protein_accession,Gene,r,is_specific,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions,excluded_transcript_types,excluded_transcript_tags
440,ENSG00000056291_ProteinCoding_1,inclusion,ENST00000308744,NPFFR2,0.4471791265064418,True,False,13,CATH-Gene3D | CATH-FunFam | CDD | PANTHER | PI...,Rhodopsin 7-helix transmembrane proteins | neu...,"- | Neuropeptide FF receptor, type 2 | G prote...",,
4029,ENSG00000146904_ProteinCoding_1,inclusion,ENST00000275815,EPHA1,0.3414524560565585,True,True,12,CATH-Gene3D | CATH-FunFam | PANTHER | PIRSF | ...,Phosphorylase Kinase; domain 1 | Transferase(P...,- | Ephrin receptor tyrosine kinases | Ephrin ...,,
5682,ENSG00000177508_ProteinCoding_1,inclusion,ENST00000329734,IRX3,-0.1988403026764951,True,False,11,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Homeodomain-like | Iroquois-class homeobox pro...,- | Homeodomain | KN homeodomain | Iroquois-cl...,,
159,ENSG00000010704_ProteinCoding_7,inclusion,ENST00000353147,HFE,-0.1190834969825156,True,True,11,CATH-Gene3D | CATH-FunFam | PANTHER | Pfam | P...,Immunoglobulins | Major histocompatibility com...,Immunoglobulin-like fold | - | Antigen-present...,,
3627,ENSG00000139880_ProteinCoding_1,inclusion,ENST00000397359,CDH24,-0.4879409185382066,True,True,9,CATH-Gene3D | CATH-FunFam | CDD | PANTHER | Pf...,Cadherins | Protocadherin beta 4 | Cadherin ta...,- | Cadherin | Cadherin-like | Cadherin-like s...,,
6595,ENSG00000243955_ProteinCoding_1,inclusion,ENST00000334575,GSTA1,0.3461300290715827,True,False,9,CATH-Gene3D | CDD | PANTHER | SFLD | SUPERFAMI...,"- | Glutaredoxin | C-terminal, alpha helical d...",- | Thioredoxin-like superfamily | Glutathione...,,
2206,ENSG00000115593_ProteinCoding_1,inclusion,ENST00000419482,SMYD1,0.2891061506351097,True,True,8,CATH-Gene3D | CATH-FunFam | CDD | PANTHER | Pf...,SET domain | Histone-lysine N-methyltransferas...,"SET domain superfamily | - | SMYD1, SET domain...",,
220,ENSG00000017260_ProteinCoding_4,inclusion,ENST00000510168,ATP2C1,0.2051146889237357,True,True,8,CATH-Gene3D | PANTHER | Pfam | Phobius | SMART...,"Calcium-transporting ATPase, transmembrane dom...","- | Cation-transporting P-type ATPase, N-termi...",,
979,ENSG00000080822_ProteinCoding_9,inclusion,ENST00000394181,CLDND1,0.4509619635694156,True,False,7,CATH-Gene3D | CATH-FunFam | PANTHER | Pfam | P...,- | Claudin domain-containing protein 1 | CLAU...,- | Claudin domain-containing protein 1 | PMP-...,,
1836,ENSG00000108387_ProteinCoding_3,inclusion,ENST00000426861,SEPTIN4,0.3501897618647627,True,True,7,CATH-Gene3D | MobiDB-lite | PANTHER | PIRSR | ...,P-loop containing nucleotide triphosphate hydr...,P-loop containing nucleoside triphosphate hydr...,,
